# Phase 2: Baseline Models & Uncalibrated Evaluation

## Goals
- Train XGBoost, LightGBM, and MLP baseline models
- Evaluate classification performance (AUROC, AUPRC, F1)
- **Crucially**: Evaluate calibration (ECE, Brier score, reliability diagrams)
- Compare i.i.d. vs temporal-split calibration
- Establish the miscalibration problem before applying fixes

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils.helpers import load_config, set_seed
from src.data.loader import get_dataset, load_creditcard, preprocess_creditcard
from src.data.splits import temporal_split_creditcard
from src.models.baselines import get_model
from src.evaluation.calibration_metrics import (
    compute_all_calibration_metrics, reliability_diagram_data
)
from src.evaluation.visualization import plot_reliability_diagram

config = load_config('../configs/config.yaml')
set_seed(config['experiment']['seed'])
print('Setup complete.')

In [ ]:
# Load and split data
df_raw = load_creditcard(f"../{config['data']['creditcard_path']}")
X, y = preprocess_creditcard(df_raw)
splits = temporal_split_creditcard(df_raw)

X_train, y_train = X.iloc[splits['train']].values, y[splits['train']]
X_cal, y_cal = X.iloc[splits['calibration']].values, y[splits['calibration']]
X_val, y_val = X.iloc[splits['val']].values, y[splits['val']]
X_test, y_test = X.iloc[splits['test']].values, y[splits['test']]

print(f'Train: {len(y_train)} (fraud: {y_train.sum()})')
print(f'Cal:   {len(y_cal)} (fraud: {y_cal.sum()})')
print(f'Val:   {len(y_val)} (fraud: {y_val.sum()})')
print(f'Test:  {len(y_test)} (fraud: {y_test.sum()})')

## Train Baseline Models

In [ ]:
models = {}
predictions = {}
results = {}

for model_name in ['xgboost', 'lightgbm']:
    print(f'\n{"="*50}')
    print(f'Training: {model_name}')
    print(f'{"="*50}')
    
    model_config = config['models'].get(model_name, {})
    model = get_model(model_name, model_config)
    model.fit(X_train, y_train, X_val, y_val)
    
    models[model_name] = model
    probs = model.predict_proba(X_test)
    predictions[model_name] = probs
    
    clf_metrics = model.evaluate(X_test, y_test)
    cal_metrics = compute_all_calibration_metrics(probs, y_test)
    results[model_name] = {**clf_metrics, **cal_metrics}
    
    print(f'AUROC: {clf_metrics["auroc"]:.4f}')
    print(f'AUPRC: {clf_metrics["auprc"]:.4f}')
    print(f'ECE:   {cal_metrics["ece"]:.4f}')
    print(f'Brier: {cal_metrics["brier_score"]:.6f}')

## Reliability Diagrams (Uncalibrated)

These diagrams are the key visualization. If models are well-calibrated, bars should align with the diagonal. Deviation = miscalibration.

In [ ]:
n_models = len(predictions)
fig, axes = plt.subplots(1, n_models, figsize=(7*n_models, 6))
if n_models == 1:
    axes = [axes]

for ax, (name, probs) in zip(axes, predictions.items()):
    plot_reliability_diagram(
        probs, y_test, n_bins=15,
        title=f'{name} (ECE={results[name]["ece"]:.4f})',
        ax=ax
    )

plt.suptitle('Uncalibrated Model Reliability Diagrams', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../results/figures/uncalibrated_reliability.png', dpi=150, bbox_inches='tight')
plt.show()

## Results Summary Table

In [ ]:
results_df = pd.DataFrame(results).T
results_df.index.name = 'Model'
print(results_df.to_markdown())
results_df.to_csv('../results/tables/baseline_results.csv')
results_df

## Key Findings

Document observations:
1. Are models overconfident or underconfident?
2. Which model has worst/best calibration?
3. How does ECE compare to AUROC — is a model with higher AUROC also better calibrated?
4. What does the reliability diagram shape tell us about the miscalibration pattern?